In [7]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [9]:
# Linear Regression Inspector: fits a proper pipeline, reports coefficients with names,
# robust statistics (p-values, CIs), diagnostics (VIF), and core metrics.
# Requires: scikit-learn, pandas, numpy, statsmodels

class LinearRegressionInspector:
    def __init__(self, fit_intercept=True, positive=False, standardize_numeric=False,
                 drop_first_categoricals=True, robust_se=True):
        self.fit_intercept = fit_intercept
        self.positive = positive
        self.standardize_numeric = standardize_numeric
        self.drop_first_categoricals = drop_first_categoricals
        self.robust_se = robust_se
        self.pipeline_ = None
        self.feature_names_ = None
        self.results_ = None
        self.metrics_ = None

    def fit(self, X: pd.DataFrame, y: pd.Series,
            num_features=None, cat_features=None):
        X = X.copy()
        y = y.copy()

        if num_features is None:
            num_features = X.select_dtypes(include=[np.number]).columns.tolist()
        if cat_features is None:
            cat_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

        num_transformer = []
        if self.standardize_numeric:
            num_transformer.append(("scaler", StandardScaler()))
        # Create a passthrough if no transforms
        num_ct = "passthrough" if not num_transformer else Pipeline(num_transformer)

        cat_ct = "passthrough"
        if len(cat_features) > 0:
            cat_ct = OneHotEncoder(drop="first" if self.drop_first_categoricals else None,
                                   handle_unknown="ignore", sparse_output=False)

        pre = ColumnTransformer(
            transformers=[
                ("num", num_ct, num_features),
                ("cat", cat_ct, cat_features)
            ],
            remainder="drop"
        )

        lin = LinearRegression(fit_intercept=self.fit_intercept, positive=self.positive)

        self.pipeline_ = Pipeline([("pre", pre), ("lin", lin)])
        self.pipeline_.fit(X, y)

        # Feature names after encoding
        numeric_names = num_features
        if len(cat_features) > 0:
            ct = self.pipeline_.named_steps["pre"]
            cat_encoder = ct.named_transformers_["cat"]
            cat_names = cat_encoder.get_feature_names_out(cat_features).tolist()
        else:
            cat_names = []
        self.feature_names_ = numeric_names + cat_names

        # Design matrix used by sklearn (post-transform)
        X_trans = self.pipeline_.named_steps["pre"].transform(X)
        # Statsmodels for inference (robust SEs by default)
        X_sm = sm.add_constant(X_trans, has_constant="add") if self.fit_intercept else X_trans
        model = sm.OLS(y, X_sm)
        if self.robust_se:
            self.results_ = model.fit(cov_type="HC3")
        else:
            self.results_ = model.fit()

        # Metrics
        y_hat = self.pipeline_.predict(X)
        n = len(y)
        p = X_trans.shape[1]
        r2 = r2_score(y, y_hat)
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1) if self.fit_intercept and n > p + 1 else np.nan
        mae = mean_absolute_error(y, y_hat)
        rmse = root_mean_squared_error(y, y_hat)

        self.metrics_ = {"R2": r2, "Adj_R2": adj_r2, "MAE": mae, "RMSE": rmse, "n": n, "p": p}

        return self

    def coef_table(self):
        if self.results_ is None:
            raise RuntimeError("Call fit() first.")
        params = self.results_.params
        conf = self.results_.conf_int()
        se = self.results_.bse
        tvals = self.results_.tvalues
        pvals = self.results_.pvalues

        names = (["Intercept"] if self.fit_intercept else []) + self.feature_names_
        df = pd.DataFrame({
            "feature": names,
            "coef": params,
            "std_err": se,
            "t": tvals,
            "p_value": pvals,
            "ci_low": conf[0],
            "ci_high": conf[1],
        })
        # Sort by absolute coefficient (ignoring intercept if present)
        if self.fit_intercept:
            order = [0] + list(1 + np.argsort(-np.abs(df["coef"].values[1:])))
            df = df.iloc[order].reset_index(drop=True)
        else:
            df = df.iloc[np.argsort(-np.abs(df["coef"].values))].reset_index(drop=True)
        return df

    def vif_table(self, X: pd.DataFrame):
        # Compute VIF on the transformed design (excluding intercept)
        if self.results_ is None:
            raise RuntimeError("Call fit() first.")
        X_trans = self.pipeline_.named_steps["pre"].transform(X)
        vif = []
        for i in range(X_trans.shape[1]):
            vif.append(variance_inflation_factor(X_trans, i))
        return pd.DataFrame({"feature": self.feature_names_, "VIF": vif})

    def report(self, X: pd.DataFrame):
        print("Metrics:", self.metrics_)
        print()
        print("Coefficients:")
        print(self.coef_table().to_string(index=False))
        print()
        try:
            vif = self.vif_table(X)
            print("VIF (multicollinearity diagnostics):")
            print(vif.to_string(index=False))
        except Exception as e:
            print("VIF not available:", e)

    def predict(self, X: pd.DataFrame):
        return self.pipeline_.predict(X)

In [10]:
# if __name__ == "__main__":
rng = np.random.default_rng(42)
n = 300
df = pd.DataFrame({
    "floor_area": rng.normal(70, 20, n).clip(20, 180),
    "bedrooms": rng.integers(1, 5, n),
    "age_years": rng.normal(40, 25, n).clip(0, 120).round(0),
    "borough": rng.choice(["Camden", "Hackney", "Lewisham", "Ealing"], n)
})
# True data-generating process
loc_w = {"Camden": 1.3, "Hackney": 1.15, "Lewisham": 0.9, "Ealing": 1.05}
base = 150_000
y_true = (base
          + 4000 * df["floor_area"]
          + 30_000 * df["bedrooms"]
          - 800 * df["age_years"]
          + 200_000 * df["borough"].map(loc_w).values)
noise = rng.normal(0, 60_000, n)
df["price"] = y_true + noise

X = df.drop(columns=["price"])
y = df["price"]

inspector = LinearRegressionInspector(
    fit_intercept=True,
    positive=False,
    standardize_numeric=False,   # set True to standardize numeric features
    drop_first_categoricals=True,
    robust_se=True               # HC3 robust standard errors
).fit(X, y, num_features=["floor_area", "bedrooms", "age_years"], cat_features=["borough"])

inspector.report(X)

Metrics: {'R2': 0.7041758193240313, 'Adj_R2': 0.6981179862726463, 'MAE': 44090.10814175449, 'RMSE': 57171.91862385974, 'n': 300, 'p': 6}

Coefficients:
         feature          coef      std_err         t       p_value        ci_low       ci_high
       Intercept 406201.810099 15674.989292 25.914009 4.630244e-148 375479.395628 436924.224570
borough_Lewisham -60326.631478  9436.510671 -6.392896  1.627730e-10 -78821.852534 -41831.410423
  borough_Ealing -35216.488585  9798.857401 -3.593938  3.257170e-04 -54421.896181 -16011.080988
        bedrooms  30089.962998  2945.364754 10.216040  1.680678e-24  24317.154159  35862.771836
 borough_Hackney -15319.726992  9881.935928 -1.550276  1.210753e-01 -34687.965509   4048.511524
      floor_area   3967.578216   171.460785 23.139858 1.839179e-118   3631.521253   4303.635179
       age_years   -933.261174   141.792578 -6.581876  4.645487e-11  -1211.169519   -655.352828

VIF (multicollinearity diagnostics):
         feature      VIF
      floor_area